# Reproduce the baseline scores

This notebook downloads the public REBUILT datasets, harmonizes them to the
project's two classes, and rescores baseline-a's committed predictions on
the three test splits. It then checks that every score matches the one
published in the README and `docs/EVALUATION.md`.

It calls no model. baseline-a runs on Roboflow's hosted API from the
workspace that trained it, so its predictions are committed under
`reports/runs/` instead, and `frc-evaluate --from-cache` scores them again
(D-023 in `docs/DECISIONS.md`). You need a free Roboflow API key for the
download only. Add it in Colab's Secrets panel (the key icon on the left)
as `ROBOFLOW_API_KEY` and give this notebook access. The datasets take about
1.5 GB on disk.

## Install

This clones the release and installs the locked core dependencies from
`requirements.txt`. The `infer` extra is left out on purpose, since it
would replace Colab's CUDA build of torch, and rescoring doesn't need it.
If Colab asks you to restart the session after the install, restart it and
carry on from the next cell.

In [ ]:
REF = "v0.1.0"
REPO = "https://github.com/nickmelamed/frc-field-gap.git"
!git clone --quiet --depth 1 --branch {REF} {REPO} /content/frc-field-gap
%cd /content/frc-field-gap
!pip install --quiet -r requirements.txt
!pip install --quiet --no-deps .

## Download and harmonize

Every command runs from the repository root, where the configs are. The key
goes into the environment and is never printed.

In [ ]:
%cd /content/frc-field-gap
import os

from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

In [ ]:
!frc-download
!frc-harmonize

## Rescore the published runs

For each published run, this scores the cached predictions again on the
freshly harmonized split and compares the result with the committed
`metrics.json`. A run refuses to start from a repository with uncommitted
files, so each new run folder is removed once it has been compared. Each
rescore takes a few minutes, mostly for the intervals.

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

from frc_xdata.runs import load_runs, predictions_path, select_runs

RUNS = Path("reports/runs")
COMPARED = [
    "scored_classes",
    "unscored_classes",
    "per_image_limit",
    "metrics",
    "units",
    "bootstrap",
]

mismatches = 0
for published in select_runs(load_runs(RUNS)):
    r = published.result
    source = predictions_path(RUNS, r).parent.name
    before = {p.name for p in RUNS.iterdir()}
    subprocess.run(
        ["frc-evaluate", r.model, r.dataset, "--split", r.split, "--from-cache", source],
        check=True,
    )
    (new_run,) = [p for p in RUNS.iterdir() if p.name not in before]
    old = json.loads((RUNS / r.run_id / "metrics.json").read_text())
    new = json.loads((new_run / "metrics.json").read_text())
    differs = [k for k in COMPARED if old[k] != new[k]]
    shutil.rmtree(new_run)
    mismatches += bool(differs)
    status = "differs in " + ", ".join(differs) if differs else "matches"
    print(f"{r.model} on {r.dataset} {r.split}: {status}")

print("All scores reproduced." if not mismatches else f"{mismatches} run(s) differ.")

## Scoring a model live (optional)

`frc-evaluate` without `--from-cache` calls the hosted model, which needs an
API key from the workspace that trained it. To score a model of your own,
add it to `reports/models.yaml` as `docs/RETRAINING.md` describes, install
the inference client alone, and run it on a harmonized split. Install
`inference-sdk` by itself, never the `infer` extra, for the reason above.

In [ ]:
# !pip install --quiet inference-sdk==1.7.2
# !frc-evaluate <model> marswars --split test --limit 10